# 01. D1 loading and data structure

- Input: 10 CSV files (AI-Hub 138)
- Spectrum: 513 bins, 0–5,120 Hz, 10 Hz step (both endpoints included)
- 49 sites, 2,710 sensors, 2019–2021
- `lrate`, `llevel`: metadata only (not used as acoustic input)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("..")) 
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import src as la
from src import config as cfg, io_utils, features, evaluation as ev, sfb, reporting as rp
cfg.apply_mpl_style(); cfg.banner()

LeakAudit | root=C:\Users\ecredible\Desktop\LeakAudit_revision1\LeakAudit | mode=FULL | seeds=10 | bootstrap=2000


In [ ]:
X1, X1_max, M1, HZ = io_utils.load_d1(force=False)

print(f"rows        : {len(M1):,}")
print(f"spectrum    : {X1.shape[1]} bins, {HZ.min()}-{HZ.max()} Hz, step {HZ[1]-HZ[0]} Hz")
print(f"MAX columns : {X1_max.shape[1]}")
print(f"sites       : {M1.site.nunique()}   sensors: {M1.sid.nunique()}")
print(f"dates       : {M1.dt.min().date()} - {M1.dt.max().date()}")
print()
print(pd.crosstab(M1.cls, M1.split, margins=True).to_string())

rows        : 70,384
spectrum    : 513 bins, 0-5120 Hz, step 10 Hz
MAX columns : 20
sites       : 49   sensors: 2710
dates       : 2019-02-27 - 2021-02-10

split   train   val    All
cls                       
in      13273  1659  14932
noise    5029   629   5658
normal  19704  2462  22166
other    7019   878   7897
out     17539  2192  19731
All     62564  7820  70384


## 1. Column inventory

0–5,120 Hz at 10 Hz step (inclusive): (5120 − 0) / 10 + 1 = 513 bins

In [3]:
per_file, inv = io_utils.d1_column_inventory()
if len(per_file):
    print(per_file.drop(columns="other_columns").to_string(index=False))
    per_file.to_csv(cfg.RESULTS / "D1_column_inventory.csv", index=False)
print()
for k, v in inv.items():
    print(f"  {k:36s} {v}")
la.save_json(inv, "D1_column_inventory.json")
rp.record("D1.n_spectral_bins", inv["n_spectral_bins"], "D1_column_inventory.json", "R1-M11,R2-15")
rp.record("D1.n_columns_total", inv["n_columns_total"], "D1_column_inventory.json", "R1-M11,R2-15",
          note=inv["source"])

                          file  n_columns  n_freq_bins  n_max_cols  n_other
      1.옥외누수(out-training).csv        539          513          20        6
    1.옥외누수(out-validation).csv        539          513          20        6
       2.옥내누수(in-training).csv        539          513          20        6
     2.옥내누수(in-validation).csv        539          513          20        6
  3.기계.전기음(noise-training).csv        539          513          20        6
3.기계.전기음(noise-validation).csv        539          513          20        6
     4.환경음(other-training).csv        539          513          20        6
   4.환경음(other-validation).csv        539          513          20        6
    5.정상음(normal-training).csv        539          513          20        6
  5.정상음(normal-validation).csv        539          513          20        6

  source                               raw CSV headers
  identical_header_in_all_files        True
  n_columns_total                      539
  n_spectral_bins    

### 1.1 Provider schema vs. distributed files

Provider schema: 5 ID/device fields + 1 label + "512" spectral columns (0–5,120 Hz) + 20 MAX columns

In [4]:
documented = dict(id_device=5, label=1, spectral_documented=512, max_cols=20)
documented["total_documented"] = sum(documented.values())
observed = dict(spectral=inv["n_spectral_bins"], max_cols=inv["n_max_columns"],
                non_spectral=inv["n_non_spectral_columns"], total=inv["n_columns_total"])
cmp_doc = dict(documented=documented, observed=observed,
               spectral_grid_inclusive=inv["n_positions_inclusive_grid"],
               reading=("documented '512' columns run from 0 Hz to 5,120 Hz at 10 Hz, which is "
                        f"{inv['n_positions_inclusive_grid']} positions; the copy has "
                        f"{inv['n_spectral_bins']} spectral bins, so the difference is the inclusive "
                        "endpoint, not a missing or extra frequency"),
               provider_note=cfg.D1_PROVIDER_DOC["spectral_columns_documented"])
for k, v in cmp_doc.items():
    print(f"  {k:24s} {v}")
la.save_json(cmp_doc, "D1_schema_vs_documentation.json")

  documented               {'id_device': 5, 'label': 1, 'spectral_documented': 512, 'max_cols': 20, 'total_documented': 538}
  observed                 {'spectral': 513, 'max_cols': 20, 'non_spectral': 6, 'total': 539}
  spectral_grid_inclusive  513
  reading                  documented '512' columns run from 0 Hz to 5,120 Hz at 10 Hz, which is 513 positions; the copy has 513 spectral bins, so the difference is the inclusive endpoint, not a missing or extra frequency
  provider_note            '512 columns', listed as item 1 = 0 Hz ... item 512 = 5,120 Hz at 10 Hz; 0-5,120 Hz with both ends is 513 positions


WindowsPath('C:/Users/ecredible/Desktop/LeakAudit_revision1/LeakAudit/results/D1_schema_vs_documentation.json')

## 2. Sensor and date structure

Unit: one record = one sensor on one date

In [5]:
per_sensor = M1.groupby("sid").agg(rows=("cls", "size"), classes=("cls", "nunique"),
                                   dates=("ldate", "nunique"), site=("site", "first"))
per_site = M1.groupby("site").agg(rows=("cls", "size"), sensors=("sid", "nunique"),
                                  classes=("cls", "nunique"))
struct = dict(
    n_rows=int(len(M1)), n_sites=int(M1.site.nunique()), n_sensors=int(M1.sid.nunique()),
    rows_per_sensor_median=float(per_sensor.rows.median()),
    rows_per_sensor_max=int(per_sensor.rows.max()),
    single_class_sensors=int((per_sensor.classes == 1).sum()),
    one_row_per_sensor_date=bool((per_sensor.rows == per_sensor.dates).all()),
    sites_by_number_of_classes=per_site.classes.value_counts().sort_index().to_dict(),
)
for k, v in struct.items():
    print(f"  {k:28s} {v}")
la.save_json(struct, "D1_structure.json")

  n_rows                       70384
  n_sites                      49
  n_sensors                    2710
  rows_per_sensor_median       18.0
  rows_per_sensor_max          153
  single_class_sensors         2710
  one_row_per_sensor_date      True
  sites_by_number_of_classes   {1: 17, 2: 10, 3: 4, 4: 16, 5: 2}


WindowsPath('C:/Users/ecredible/Desktop/LeakAudit_revision1/LeakAudit/results/D1_structure.json')

## 3. Table 1 (D1): unit of analysis and majority rate

In [6]:
y5 = M1.cls.to_numpy()
tasks = {
    "T5_five_class": y5,
    "TL_leak_vs_nonleak": np.where(np.isin(y5, ["out", "in"]), "leak", "nonleak"),
    "TR_out_vs_in": y5[np.isin(y5, ["out", "in"])],
    "TA_alarm_vs_normal": np.where(y5 == "normal", "normal", "alarm"),
}
rows = []
for t, yt in tasks.items():
    vc = pd.Series(yt).value_counts()
    rows.append(dict(dataset="D1", task=t, definition=cfg.D1_TASKS[t],
                     unit="record (one sensor on one date)", n=int(len(yt)),
                     class_counts="; ".join(f"{k}={v}" for k, v in vc.items()),
                     majority_rate=float(vc.max() / vc.sum()),
                     groups=f"{M1.site.nunique()} sites / {M1.sid.nunique()} sensors"))
t1 = pd.DataFrame(rows)
t1.to_csv(cfg.RESULTS / "table1_D1.csv", index=False)
print(t1.drop(columns="definition").to_string(index=False))
for _, r in t1.iterrows():
    rp.record(f"D1.majority.{r.task}", r.majority_rate, "table1_D1.csv", "R1-minor(Table1)")

dataset               task                            unit     n                                              class_counts  majority_rate                  groups
     D1      T5_five_class record (one sensor on one date) 70384 normal=22166; out=19731; in=14932; other=7897; noise=5658       0.314930 49 sites / 2710 sensors
     D1 TL_leak_vs_nonleak record (one sensor on one date) 70384                                 nonleak=35721; leak=34663       0.507516 49 sites / 2710 sensors
     D1       TR_out_vs_in record (one sensor on one date) 34663                                       out=19731; in=14932       0.569224 49 sites / 2710 sensors
     D1 TA_alarm_vs_normal record (one sensor on one date) 70384                                 alarm=48218; normal=22166       0.685070 49 sites / 2710 sensors


In [7]:
print("cached files:")
for p in sorted(cfg.CACHE.glob("d1_*")):
    print("  ", p.name, f"{p.stat().st_size/1e6:.1f} MB")

cached files:
   d1_hz.npy 0.0 MB
   d1_max.npy 5.6 MB
   d1_meta.csv 5.4 MB
   d1_spec.npy 144.4 MB
